In [1]:
# %%
import pandas as pd
import numpy as np

from pathlib import Path
from scipy.stats import chi2_contingency

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [2]:
# %%
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "model_matrix.parquet"

In [3]:
# %%
df = pd.read_parquet(DATA_PATH)

df.shape, df.columns.tolist()

((535391, 39),
 ['browser_desktop_web',
  'os_name_clean_Android',
  'os_name_clean_Chrome OS',
  'os_name_clean_Other',
  'os_name_clean_Windows',
  'os_name_clean_macOS',
  'sub_region_East North Central',
  'sub_region_East South Central',
  'sub_region_Middle Atlantic',
  'sub_region_Mountain',
  'sub_region_New England',
  'sub_region_Pacific',
  'sub_region_West North Central',
  'sub_region_West South Central',
  'test_engagement_state_Test Completed',
  'test_engagement_state_Test Started Only',
  'tests_taken_count_bucket_1',
  'tests_taken_count_bucket_2–3',
  'tests_taken_count_bucket_4+',
  'tests_completed_count_bucket_1',
  'tests_completed_count_bucket_2+',
  'first_test_domain_Addiction & Compulsive Behavior',
  'first_test_domain_Anxiety & Stress',
  'first_test_domain_Life, Work & Physical Health',
  'first_test_domain_Mood & Depression',
  'first_test_domain_Neurodevelopmental & Cognitive',
  'first_test_domain_Personality Disorders & Traits',
  'first_test_domain_Se

In [4]:
# %%
assert "has_signup" in df.columns
assert df["has_signup"].dropna().isin([0, 1, True, False]).all()

In [5]:
# %%
TARGET = "has_signup"
ID_COLS = ["visitor_key"]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET]
]

len(FEATURE_COLS)

37

In [6]:
# %%
non_binary = [
    c for c in FEATURE_COLS
    if not set(df[c].dropna().unique()).issubset({0, 1})
]

non_binary

[]

In [7]:
# %%
def chi_square_binary_feature(df, feature, target="has_signup"):
    """
    Runs a Chi-square test between a binary feature and binary target.
    Returns stats + interpretable metrics.
    """
    contingency = pd.crosstab(df[feature], df[target])

    # Ensure full 2x2 table
    if contingency.shape != (2, 2):
        return None

    chi2, p_value, _, _ = chi2_contingency(contingency)

    signup_rate_when_present = (
        contingency.loc[1, 1] / contingency.loc[1].sum()
        if 1 in contingency.index else np.nan
    )

    support = contingency.loc[1].sum() if 1 in contingency.index else 0

    return {
        "feature": feature,
        "chi2": chi2,
        "p_value": p_value,
        "signup_rate_when_present": signup_rate_when_present,
        "support": support
    }

In [8]:
# %%
results = []

for feature in FEATURE_COLS:
    res = chi_square_binary_feature(df, feature, TARGET)
    if res:
        results.append(res)

results_df = pd.DataFrame(results)
results_df.shape

(37, 5)

In [9]:
# %%
results_df = results_df.sort_values("p_value")
results_df.head(20)

,feature,chi2,p_value,signup_rate_when_present,support
14,test_engagement_state_Test Completed,2080.925450,0.000000e+00,0.020321,425185
19,tests_completed_count_bucket_1,2583.690581,0.000000e+00,0.022265,366807
15,test_engagement_state_Test Started Only,1410.245778,1.247399e-308,0.000503,77550
32,first_completed_test_domain_Mood & Depression,857.037037,2.146382e-188,0.027191,95326
16,tests_taken_count_bucket_1,667.350452,3.765349e-147,0.018884,401457
24,first_test_domain_Mood & Depression,498.028958,2.551707e-110,0.023934,109218
0,browser_desktop_web,295.630741,2.949230e-66,0.009311,82269
20,tests_completed_count_bucket_2+,274.048621,1.487832e-61,0.008102,58378
18,tests_taken_count_bucket_4+,223.219167,1.795658e-50,0.006562,35354
1,os_name_clean_Android,132.183210,1.364291e-30,0.019533,147286


In [10]:
# %%
results_df["significant_0_05"] = results_df["p_value"] < 0.05
results_df["significant_0_01"] = results_df["p_value"] < 0.01

results_df["significant_0_05"].value_counts()

significant_0_05
True     28
False     9
Name: count, dtype: int64

In [13]:
# %%
MIN_SUPPORT = 500  # adjust as needed

filtered = results_df[
    (results_df["support"] >= MIN_SUPPORT) &
    (results_df["p_value"] < 0.05)
]

filtered

,feature,chi2,p_value,signup_rate_when_present,support,significant_0_05,significant_0_01
14,test_engagement_state_Test Completed,2080.925450,0.000000e+00,0.020321,425185,True,True
19,tests_completed_count_bucket_1,2583.690581,0.000000e+00,0.022265,366807,True,True
15,test_engagement_state_Test Started Only,1410.245778,1.247399e-308,0.000503,77550,True,True
32,first_completed_test_domain_Mood & Depression,857.037037,2.146382e-188,0.027191,95326,True,True
16,tests_taken_count_bucket_1,667.350452,3.765349e-147,0.018884,401457,True,True
24,first_test_domain_Mood & Depression,498.028958,2.551707e-110,0.023934,109218,True,True
0,browser_desktop_web,295.630741,2.949230e-66,0.009311,82269,True,True
20,tests_completed_count_bucket_2+,274.048621,1.487832e-61,0.008102,58378,True,True
18,tests_taken_count_bucket_4+,223.219167,1.795658e-50,0.006562,35354,True,True
1,os_name_clean_Android,132.183210,1.364291e-30,0.019533,147286,True,True


In [12]:
# %%
filtered.assign(
    feature_group=filtered["feature"].str.split("_").str[0]
).groupby("feature_group").size().sort_values(ascending=False)

feature_group
first      10
os          5
tests       5
sub         5
test        2
browser     1
dtype: int64